# Uscite ed entrate dall'S&P 500: momentum o reversal?

**Idea** (da Conrad & Yavuz, *Momentum and Reversal*):
- le azioni che **escono** dall'S&P 500 dovrebbero essere **loser** (hanno perso nell'ultimo anno);
- quelle che **entrano** dovrebbero essere **winner**;
- nei 6 mesi dopo l'evento: continuano (momentum) o tornano indietro (reversal)?

**Domande**
1. Le uscite sono loser e le entrate winner?
2. Che percentuale fa reversal nei 6 mesi successivi?
3. Quanto rende un portafoglio che compra le uscite (o le entrate)?
4. Le uscite si comportano diversamente dai loser che **restano** nell'indice?

**Convenzioni**
- t = mese in cui l'evento diventa effettivo.
- Rendimento di formazione = mesi da t−12 a t−2 (si salta t−1, come nel momentum classico).
- Periodo di detenzione = mesi da t+1 a t+6.
- Campione: eventi dal 1980 al 2022 (il file WRDS dei membri finisce a fine 2022).

Il notebook legge solo i file in `Data/` e non modifica niente.

## 0. Impostazioni

In [ ]:
from pathlib import Path
import io, zipfile

import numpy as np
import pandas as pd
import statsmodels.api as sm

pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", None)

# cartella Data: funziona sia se apri il notebook dalla cartella principale sia da notebooks/
DATA = next(p / "Data" for p in [Path.cwd(), *Path.cwd().parents] if (p / "Data").is_dir())

HOLD = 6                 # mesi di detenzione
LOSER, WINNER = 0.30, 0.70  # loser = 30% più basso, winner = 30% più alto
FIRST_YEAR, LAST_YEAR = 1980, 2022
MATCH_BAND = 0.05        # Q4: tolleranza nell'abbinare uscite e "rimasti"


def find_file(*names):
    """Restituisce il primo file esistente tra i nomi indicati (nome descrittivo o nome WRDS)."""
    for name in names:
        if (DATA / name).is_file():
            return DATA / name
    raise FileNotFoundError(f"Nessuno di questi file è in {DATA}: {names}")


print("Cartella dati:", DATA)

## 1. Membri dell'S&P 500

Il file WRDS ha una riga per ogni periodo ("spell") in cui un'azienda è stata nell'indice: PERMNO, data di ingresso, data di uscita.

In [ ]:
spells = pd.read_csv(find_file("sp500_constituent_changes.csv"), parse_dates=["SP500 Start", "SP500 End"])
spells = spells.rename(columns={"PERMNO": "permno", "SP500 Start": "start", "SP500 End": "end"})
spells = spells[["permno", "start", "end"]].drop_duplicates().sort_values(["permno", "start"])
END_OF_FILE = spells["end"].max()   # ultima data del file: chi ha questa data di fine è ancora dentro

print(f"{len(spells)} periodi, {spells['permno'].nunique()} aziende, file fino al {END_OF_FILE.date()}")
spells.head()

## 2. Rendimenti mensili CRSP

Teniamo solo le aziende che sono state almeno una volta nell'S&P 500: sono le uniche che ci servono.
Costruiamo una tabella "larga": una riga per mese, una colonna per azienda.

Nota: nel formato CRSP v2 `MthRet` include già il rendimento di delisting.

In [ ]:
monthly_file = find_file("crsp_monthly_stock_returns.csv", "jhvzegjsguw3gw3k.csv")
crsp = pd.read_csv(monthly_file, usecols=["PERMNO", "MthCalDt", "MthRet"], parse_dates=["MthCalDt"])
crsp = crsp[crsp["PERMNO"].isin(spells["permno"])].drop_duplicates(["PERMNO", "MthCalDt"])
crsp["month"] = crsp["MthCalDt"].dt.to_period("M")

ret = crsp.pivot(index="month", columns="PERMNO", values="MthRet").sort_index()
ret.columns.name = "permno"
months = ret.index
print(f"Rendimenti: {ret.shape[0]} mesi x {ret.shape[1]} aziende, dal {months[0]} al {months[-1]}")

In [ ]:
# rendimento del mercato (CRSP value-weighted, dividendi inclusi)
index_file = find_file("crsp_monthly_index_returns.csv", "pc8rgx9sahq2tuh6.csv")
mkt = pd.read_csv(index_file)
mkt.columns = mkt.columns.str.lower()
mkt["month"] = pd.to_datetime(mkt["mthcaldt"]).dt.to_period("M")
mkt = mkt.set_index("month")["vwretd"].reindex(months)
mkt.tail()

## 3. Chi è nell'indice ogni mese

`member` è una tabella vero/falso: `member.loc[mese, azienda]` è vero se l'azienda è nell'indice alla fine di quel mese.

In [ ]:
member = pd.DataFrame(False, index=months, columns=ret.columns)
month_end = months.to_timestamp(how="end")
for s in spells.itertuples():
    if s.permno in member.columns:
        member.loc[(month_end >= s.start) & (month_end <= s.end), s.permno] = True

print("Numero di membri a fine anno (dovrebbe essere circa 500):")
member.sum(axis=1)[months.month == 12].iloc[::10]

## 4. Rendimento di formazione e rendimento nei 6 mesi dopo

- `mom.loc[t]` = rendimento cumulato da t−12 a t−2 (servono tutti gli 11 mesi).
- `hold.loc[t]` = rendimento cumulato da t+1 a t+6. Se l'azienda esce dalla borsa prima, i soldi restano in cassa (rendimento 0).
- `hold_ar` = `hold` meno il rendimento del mercato negli stessi 6 mesi (rendimento "aggiustato per il mercato").

In [ ]:
log_ret = np.log1p(ret)
mom = np.expm1(log_ret.shift(2).rolling(11, min_periods=11).sum())

log_ret_cash = log_ret.fillna(0)                                    # dopo il delisting: cassa
hold = np.expm1(log_ret_cash.rolling(HOLD).sum().shift(-HOLD))
hold_mkt = np.expm1(np.log1p(mkt).rolling(HOLD).sum().shift(-HOLD))
hold_ar = hold.sub(hold_mkt, axis=0)

# percentile del rendimento di formazione tra i membri dell'indice a t-1 (0 = peggiore, 1 = migliore)
peer_mom = mom.where(member.shift(1, fill_value=False))


def pct_among_members(month, value):
    peers = peer_mom.loc[month].dropna()
    return (peers < value).mean() if len(peers) and pd.notna(value) else np.nan

## 5. Gli eventi: entrate e uscite

- **Entrata** = inizio di un periodo nell'indice.
- **Uscita** = fine di un periodo, ma solo se l'azienda non rientra entro 5 giorni (a volte il file spezza un periodo in due).

Filtri:
- teniamo solo gli eventi con il rendimento di formazione completo. Questo esclude da solo spin-off e nuove quotazioni, che non hanno 12 mesi di storia;
- per le uscite, l'azienda deve essere ancora quotata nel mese t+1. Altrimenti è uscita perché acquisita, non perché è scesa.

In [ ]:
next_start = spells.groupby("permno")["start"].shift(-1)
real_exit = spells["end"].lt(END_OF_FILE) & ~(next_start - spells["end"]).dt.days.le(5)

entries = spells[["permno", "start"]].rename(columns={"start": "date"}).assign(type="entry")
exits = spells.loc[real_exit, ["permno", "end"]].rename(columns={"end": "date"}).assign(type="exit")
ev = pd.concat([entries, exits], ignore_index=True)
ev["month"] = ev["date"].dt.to_period("M")
ev = ev[ev["month"].dt.year.between(FIRST_YEAR, LAST_YEAR) & ev["permno"].isin(ret.columns)]


def lookup(table, row, shift=0):
    m = row["month"] + shift
    return table.at[m, row["permno"]] if m in table.index else np.nan


ev["mom"] = ev.apply(lambda r: lookup(mom, r), axis=1)
ev["ret_t1"] = ev.apply(lambda r: lookup(ret, r, 1), axis=1)
ev["hold"] = ev.apply(lambda r: lookup(hold, r), axis=1)
ev["hold_ar"] = ev.apply(lambda r: lookup(hold_ar, r), axis=1)
ev["pct"] = ev.apply(lambda r: pct_among_members(r["month"], r["mom"]), axis=1)

n_before = ev.groupby("type").size()
ev = ev[ev["mom"].notna() & (ev["type"].eq("entry") | ev["ret_t1"].notna())].reset_index(drop=True)
ev["period"] = pd.cut(ev["month"].dt.year, [1979, 1999, 2009, 2017, 2022],
                      labels=["1980–1999", "2000–2009", "2010–2017", "2018–2022"])

pd.DataFrame({"eventi totali": n_before, "dopo i filtri": ev.groupby("type").size()})

## Q1. Le uscite sono loser e le entrate winner?

Confrontiamo il rendimento di formazione di ogni evento con quello dei membri dell'indice nello stesso mese.

In [ ]:
def q1_table(g):
    return pd.Series({
        "N": len(g),
        "rendimento formazione medio": g["mom"].mean(),
        "rendimento formazione mediano": g["mom"].median(),
        "percentile medio": g["pct"].mean(),
        "% loser (30% più basso)": 100 * (g["pct"] <= LOSER).mean(),
        "% winner (30% più alto)": 100 * (g["pct"] >= WINNER).mean(),
    })


display(ev.groupby("type").apply(q1_table, include_groups=False).T)
display(ev.groupby(["type", "period"], observed=True).apply(q1_table, include_groups=False))

## Q2. Reversal o continuation nei 6 mesi dopo?

- un'**uscita** fa reversal se batte il mercato (`hold_ar > 0`);
- un'**entrata** fa reversal se fa peggio del mercato (`hold_ar < 0`).

Guardiamo la percentuale, ma anche **media** e **mediana**: se la media è molto più alta della mediana, il risultato dipende da pochi casi estremi.

In [ ]:
ev["reversal"] = np.where(ev["type"].eq("exit"), ev["hold_ar"] > 0, ev["hold_ar"] < 0)
ev["loser"] = ev["pct"] <= LOSER
ev["winner"] = ev["pct"] >= WINNER


def q2_table(g):
    return pd.Series({
        "N": len(g),
        "% reversal": 100 * g["reversal"].mean(),
        "rendimento 6m medio aggiustato": g["hold_ar"].mean(),
        "rendimento 6m mediano aggiustato": g["hold_ar"].median(),
        "t-stat della media": g["hold_ar"].mean() / g["hold_ar"].std() * np.sqrt(len(g)),
    })


is_exit, is_entry = ev["type"].eq("exit"), ev["type"].eq("entry")
display(pd.DataFrame({
    "tutte le uscite": q2_table(ev[is_exit]),
    "uscite loser": q2_table(ev[is_exit & ev["loser"]]),
    "tutte le entrate": q2_table(ev[is_entry]),
    "entrate winner": q2_table(ev[is_entry & ev["winner"]]),
}))
display(ev.groupby(["type", "period"], observed=True).apply(q2_table, include_groups=False))

## Q3. Portafogli

Ogni mese il portafoglio contiene, con pesi uguali, tutte le azioni che hanno avuto l'evento nei 6 mesi precedenti.

Per giudicarlo usiamo:
- **rendimento medio** e t-stat;
- **alpha CAPM** e **alpha Fama-French 5 fattori + momentum**, cioè il rendimento che i fattori di rischio noti non spiegano;
- errori standard di Newey-West, perché i portafogli si sovrappongono da un mese all'altro.

I fattori si leggono dagli zip di Kenneth French in `Data/`.

In [ ]:
def read_french(zip_name, columns):
    """Legge un file zip di Kenneth French e tiene solo le righe mensili (AAAAMM)."""
    with zipfile.ZipFile(DATA / zip_name) as z:
        lines = z.read(z.namelist()[0]).decode("latin1").splitlines()
    header = next(i for i, l in enumerate(lines) if set(columns) <= {c.strip() for c in l.split(",")})
    rows = []
    for l in lines[header + 1:]:
        parts = [x.strip() for x in l.split(",")]
        if not (len(parts[0]) == 6 and parts[0].isdigit()):
            break
        rows.append(parts)
    names = ["month"] + [c.strip() for c in lines[header].split(",")[1:]]
    df = pd.DataFrame(rows, columns=names[:len(rows[0])])
    df["month"] = pd.PeriodIndex(df["month"], freq="M")
    return df.set_index("month")[columns].astype(float) / 100


factors = read_french("F-F_Research_Data_5_Factors_2x3_CSV.zip", ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]).join(
    read_french("F-F_Momentum_Factor_CSV.zip", ["Mom"]), how="inner")
factors.tail()

In [ ]:
def portfolio(event_rows):
    """Rendimento mensile medio delle azioni nei 6 mesi dopo il loro evento."""
    held = pd.DataFrame(False, index=months, columns=ret.columns)
    for r in event_rows.itertuples():
        held.loc[(months > r.month) & (months <= r.month + HOLD), r.permno] = True
    r = ret.where(held)
    return pd.DataFrame({"ret": r.mean(axis=1), "n": r.notna().sum(axis=1)})


def evaluate(r, is_spread=False):
    """Rendimento medio, alpha CAPM e alpha FF5+Mom con t-stat di Newey-West."""
    d = pd.concat([r.rename("r"), factors], axis=1, join="inner").dropna()
    d = d[d.index.year <= LAST_YEAR + 1]
    y = d["r"] if is_spread else d["r"] - d["RF"]      # uno spread long-short è già un rendimento in eccesso
    fit = lambda X: sm.OLS(y, sm.add_constant(X)).fit(cov_type="HAC", cov_kwds={"maxlags": HOLD})
    mean = sm.OLS(y, np.ones(len(y))).fit(cov_type="HAC", cov_kwds={"maxlags": HOLD})
    capm = fit(d[["Mkt-RF"]])
    ff = fit(d[["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom"]])
    return pd.Series({
        "mesi": len(d),
        "rend. medio %/mese": 100 * d["r"].mean(),
        "t (rend. in eccesso)": mean.tvalues.iloc[0],
        "vol annua %": 100 * d["r"].std() * np.sqrt(12),
        "alpha CAPM %/mese": 100 * capm.params["const"], "t CAPM": capm.tvalues["const"],
        "alpha FF5+Mom %/mese": 100 * ff.params["const"], "t FF5+Mom": ff.tvalues["const"],
        "beta mercato": ff.params["Mkt-RF"], "beta SMB": ff.params["SMB"],
        "beta HML": ff.params["HML"], "beta Mom": ff.params["Mom"],
    })


ports = {
    "uscite": portfolio(ev[is_exit]),
    "uscite loser": portfolio(ev[is_exit & ev["loser"]]),
    "entrate": portfolio(ev[is_entry]),
}
spread = (ports["uscite"]["ret"] - ports["entrate"]["ret"]).dropna()

q3 = pd.DataFrame({name: evaluate(p["ret"].dropna()) for name, p in ports.items()})
q3["long uscite − short entrate"] = evaluate(spread, is_spread=True)
display(q3.T)
print("Numero medio di azioni in portafoglio:", {k: round(p.loc[p["n"] > 0, "n"].mean(), 1) for k, p in ports.items()})

## Q4. Gruppo di controllo: loser che escono vs loser che restano

Se l'uscita dall'indice conta davvero, le uscite devono comportarsi **diversamente** da loser simili che restano nell'indice.

**Chi sono i "rimasti"**: aziende nell'indice a fine t−1 e **ancora nell'indice a fine t**.
Attenzione: non usiamo "ancora nell'indice 6 mesi dopo". Sarebbe guardare il futuro, perché i loser che continuano a scendere vengono tolti più avanti, e il gruppo di controllo risulterebbe migliore di quanto è.

**Confronto evento per evento**: ogni uscita viene abbinata ai rimasti dello stesso mese con un percentile di formazione simile (±0,05).

In [ ]:
exit_now = pd.DataFrame(False, index=months, columns=ret.columns)
for r in ev[is_exit].itertuples():
    exit_now.at[r.month, r.permno] = True

stayer = member.shift(1, fill_value=False) & member & ~exit_now
stayer_pct = peer_mom.rank(axis=1, method="min").sub(1).div(peer_mom.notna().sum(axis=1), axis=0)  # stessa regola "quota sotto"


def control(row):
    m = row["month"]
    ok = stayer.loc[m] & (stayer_pct.loc[m] - row["pct"]).abs().le(MATCH_BAND) & hold_ar.loc[m].notna()
    ctrl = hold_ar.loc[m][ok]
    return pd.Series({"n controlli": len(ctrl), "ar controlli": ctrl.mean(), "% reversal controlli": 100 * (ctrl > 0).mean()})


ex = ev[is_exit].copy()
ex = ex.join(ex.apply(control, axis=1))
ex = ex[ex["n controlli"] > 0]
ex["differenza"] = ex["hold_ar"] - ex["ar controlli"]


def q4_table(g):
    d = g["differenza"]
    return pd.Series({
        "N uscite": len(g),
        "controlli per uscita (media)": g["n controlli"].mean(),
        "% reversal uscite": 100 * (g["hold_ar"] > 0).mean(),
        "% reversal rimasti": g["% reversal controlli"].mean(),
        "rend. 6m aggiustato uscite": g["hold_ar"].mean(),
        "rend. 6m aggiustato rimasti": g["ar controlli"].mean(),
        "differenza media": d.mean(),
        "differenza mediana": d.median(),
        "t-stat differenza": d.mean() / d.std() * np.sqrt(len(d)),
    })


display(pd.DataFrame({"tutte le uscite": q4_table(ex), "uscite loser": q4_table(ex[ex["loser"]])}))
display(ex[ex["loser"]].groupby("period", observed=True).apply(q4_table, include_groups=False))

**Confronto con i portafogli**: ogni mese compriamo tutti i loser rimasti (30% più basso tra i membri) e li teniamo 6 mesi, come le uscite. Poi guardiamo lo spread long uscite loser − short loser rimasti.

In [ ]:
loser_stayer = stayer & stayer_pct.le(LOSER)
held = pd.DataFrame(False, index=months, columns=ret.columns)
for k in range(1, HOLD + 1):
    held |= loser_stayer.shift(k, fill_value=False)
r_ls = ret.where(held)
ports["loser rimasti"] = pd.DataFrame({"ret": r_ls.mean(axis=1), "n": r_ls.notna().sum(axis=1)})

spread_q4 = (ports["uscite loser"]["ret"] - ports["loser rimasti"]["ret"]).dropna()
q4 = pd.DataFrame({
    "uscite loser": evaluate(ports["uscite loser"]["ret"].dropna()),
    "loser rimasti": evaluate(ports["loser rimasti"]["ret"].dropna()),
    "long uscite loser − short loser rimasti": evaluate(spread_q4, is_spread=True),
})
display(q4.T)
print("Loser rimasti in portafoglio, media:", round(ports["loser rimasti"].loc[lambda p: p["n"] > 0, "n"].mean(), 1))